# From FNN to RNN — Why Sequence Architecture Matters

## The Problem with FNN on Sequential Data

A FNN takes a flat vector as input. To use past information, you manually build a window of lagged returns:
```python
input = [return_t-5, return_t-4, return_t-3, return_t-2, return_t-1]  →  predict volatility_t
```

This works, but has three fundamental limitations.

---

## Limitation 1 — Order Blindness

The FNN assigns an independent weight to each lag. It has no built-in understanding that lag 1 and lag 2 are neighbors in time, or that one happened before the other. The temporal ordering only exists because you constructed the vector that way.

**Proof:** shuffle the elements of the input vector randomly — the FNN produces nearly identical results. An RNN would break completely.

---

## Limitation 2 — Fixed Hard Cutoff

You must decide upfront how many lags to include. If volatility clustering from 45 days ago is relevant today, a window of 30 days will never capture it — no matter how well you train the model.

---

## Limitation 3 — Parameter Explosion

Every extra lag adds a full set of input weights. With 60 days and 5 features, the first layer already has 300 weights just for the inputs. Doubling the window doubles the parameters.

---

## What the RNN Changes

Instead of consuming a flat vector, the RNN processes one time step at a time and maintains a **hidden state** — a compressed memory of everything seen so far.

The same weight matrix is reused at every step — so the number of parameters does not grow with sequence length.

---

## Why This Matters for Volatility Forecasting

Volatility clustering means high volatility today predicts high volatility tomorrow — and this effect can persist for **weeks or months**. A FNN with a 10-day window is structurally blind to anything beyond day 10.

The RNN hidden state accumulates this context automatically. A spike 3 weeks ago influences h_t-20, which flows into h_t-19, all the way to today's prediction. The network decides what to keep and what to forget — rather than you deciding upfront with a fixed window.

---

## What RNN Does Not Solve — The Vanishing Gradient

In practice the hidden state cannot carry information indefinitely. During backpropagation through many time steps, gradients shrink exponentially — the network effectively forgets anything beyond ~10-20 steps. This is the **vanishing gradient problem**, and it is the motivation for the LSTM.

In [10]:
import torch
import torch.nn as nn
import sys
sys.path.append('src')
from data_utils import *
from dl_utils.Operation import WeightMultiply
from dl_utils.NumberWithGrad import NumberWithGrad
from sklearn.model_selection import train_test_split
from dl_utils.Rnn import *
import pandas as pd

Custom Implementation

In [11]:
a = NumberWithGrad(3)
b = a * 4
c = b + 5


With Torch

In [12]:
a = torch.tensor(3.0, requires_grad=True)
b = a * 4
c = b + 3
c.backward()
print(a.grad)

tensor(4.)


In [13]:
path_data = r"./data/df_sp_500_log_ret.csv"
df = pd.read_csv(path_data, index_col= "Date")

X,y = build_dataset_abs_returns_sequential(df, 10 )
X, y, scaler_x, scaler_y = scale_dataset_3d(X,y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=80718)


In [14]:
X_train.shape

(5833, 10, 1)

#

In [15]:
INPUT_SIZE = X_train.shape[2]
OUTPUT_SIZE = y_train.shape[-1]
SEQUENCE_LENGTH = 10

HIDDEN_SIZE = 32
NUM_LAYER = 1 #number of Recurrent node depth
NODE_ACTIVATION = "tanh"
DROPOUT = 0.0
BIAS = True
BATCH_NORM = False
BIDIRECTIONAL = False

HEAD_ACTIVATION = "relu"
HEAD_DROPOUT = 0
HEAD_BIAS = True
HEAD_WEIGHT_INIT = 'xavier_uniform'
HEAD_BIAS_INIT = "zeros"
HEAD_BATCH_NORM = False

nn_head_1 = Dense(HIDDEN_SIZE, 
                OUTPUT_SIZE, 
                HEAD_ACTIVATION, 
                HEAD_DROPOUT,
                HEAD_BIAS,
                HEAD_WEIGHT_INIT,
                HEAD_BIAS_INIT, 
                HEAD_BATCH_NORM
                )

Rnn_simple = RNN(input_size = INPUT_SIZE,
                hidden_size = HIDDEN_SIZE,
                num_layers=1,
                nonlinearity = NODE_ACTIVATION,
                rnn_dropout = DROPOUT,
                bidirectional=BIDIRECTIONAL,
                head = [nn_head_1]
                 )

optimizer = torch.optim.Adam(Rnn_simple.parameters(), lr=0.001)

LOSS = 'mse'
OPTIMIZER = "adam"
LR = 0.001
BATCH_SIZE = 32
N_EPOCH = 100
SHUFFLE = True
GRADIENT_CLIPPING = 1
EARLY_STOPPING = 10
VERBOSE = 10

trainer = Trainer(Rnn_simple,
                LOSS,
                OPTIMIZER,
                LR,
                BATCH_SIZE,
                N_EPOCH,
                SHUFFLE,
                GRADIENT_CLIPPING,
                EARLY_STOPPING,
                VERBOSE)
trainer.fit(X_train, y_train, X_test, y_test)

epoch    0  train: 0.988200  test: 0.863085
epoch   10  train: 0.850041  test: 0.834096
epoch   20  train: 0.789987  test: 0.817644
epoch   30  train: 0.765998  test: 0.813331
early stopping at epoch 32 — best test loss: 0.794320


In [16]:
y_train

array([[-0.37479099],
       [-0.89156758],
       [-0.86098174],
       ...,
       [-0.46058199],
       [-0.31937928],
       [-0.72539904]], shape=(5833, 1))

In [17]:
from torch.utils.data import DataLoader



In [18]:
torch.tensor(y_train)

tensor([[-0.3748],
        [-0.8916],
        [-0.8610],
        ...,
        [-0.4606],
        [-0.3194],
        [-0.7254]], dtype=torch.float64)